# CapFinch — Synthetic Omnichannel Data Generator

Builds a small, internally-consistent fake dataset (6 tables) to develop and test the
analytics / KPI pipeline **before** the e-commerce site launches.

CapFinch has run a physical store on **Square POS** for a while and is now launching its
first website, so transactions arrive from two sources. Both land in a **single `orders`
table** tagged with a `channel` flag, with channel-specific columns left null on the other
side — the same way Square's Orders API mixes in-store and Square Online sales.

**Two anchor keys**
- `transaction_id` — PK of `orders` (one row per completed sale → revenue, AOV)
- `customer_id` — PK of `customers` (one row per person → repeat rate, demographics).
  **Nullable on `orders`**: most walk-ins are anonymous, and some online buyers check out as guests.

**Relationship chain**
```
customers ─┬─ sessions ── events          (online only)
           └─ orders ── order_items ── products
```

**Realism rules baked in**
| Rule | Detail |
|---|---|
| Store hours | Open every day 10:00–18:00; no in-store order outside that window |
| In-store time-of-day | Slow morning, lunch bump, late-afternoon peak |
| In-store day-of-week | Saturday busiest, Mon/Tue slowest |
| Online time-of-day | 24/7 with a 19:00–22:00 peak and a 02:00–06:00 trough |
| Timeline | In-store history predates launch; online orders start on `LAUNCH_DATE` |
| Identity capture | ~68% of in-store orders anonymous; ~12% of online orders are guests |
| Payment mix | Cash/gift card only in store; PayPal only online |
| Basket size | Larger in store, mostly single-item online |

**KPI targets (funnel metrics are online-only by construction)**
| Metric | Target |
|---|---|
| Conversion rate (online orders / sessions) | ~2% |
| Cart abandonment (1 − completed/carts) | ~68% |
| Repeat purchase (customers with ≥2 attributable orders) | ~20% |

Everything is produced as a pandas `DataFrame` first (no CSV yet). Scale up later by raising
`N_CUSTOMERS` — the ratios above are preserved. A commented CSV-export cell is at the bottom.


In [75]:
# Dependencies live in .venv — install with:  .venv/bin/python -m pip install -r requirements.txt
# (marimo must be launched from that same venv, or it won't see these packages)

import random
from itertools import count

import numpy as np
import pandas as pd
from faker import Faker

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker("en_US")
Faker.seed(SEED)

pd.set_option("display.max_columns", None)
print("Libraries ready.")


Libraries ready.


## Config, timeline & KPI targets

`N_CUSTOMERS` is the single knob for dataset size. Repeat buyers get a 2nd order, so
*attributable* orders ≈ `N_CUSTOMERS × (1 + repeat_rate)`; anonymous walk-ins and guest
checkouts are then layered on top to hit the identity-capture rates. Session and cart counts
are derived from the **online** orders only, so conversion (~2%) and abandonment (~68%) hold
at any scale.


In [ ]:
# ---- Scale (raise this to grow every table; KPI ratios are preserved) ----
N_CUSTOMERS = 15
N_PRODUCTS = 38  # catalog holds 38 SKUs across 5 categories and 19 subcategories

# ---- Timeline: in-store history predates the website ----
TODAY = pd.Timestamp("2026-08-31").normalize()
HISTORY_START = TODAY - pd.DateOffset(months=18)
LAUNCH_DATE = (TODAY - pd.DateOffset(months=3)).normalize()  # first possible online order

# ---- KPI targets ----
TARGET_CONVERSION = 0.02   # online orders / total sessions
TARGET_ABANDONMENT = 0.68  # 1 - completed_carts / carts_created
TARGET_REPEAT = 0.20       # share of customers with >= 2 orders

# ---- Channel mix & identity capture ----
P_ONLINE_ATTRIBUTABLE = 0.40  # of known-customer orders, share placed on the website
INSTORE_ANON_RATE = 0.68      # in-store orders with no customer_id (no receipt captured)
ONLINE_GUEST_RATE = 0.12      # online orders checked out as a guest
BIRTHDATE_CAPTURE_RATE = 0.45  # share of customers who actually hand over a birthday
GIFT_SHIP_RATE = 0.20          # online orders shipped somewhere other than the buyer's address

# ---- Geography: a boutique's customers cluster near the store (Richmond, VA) ----
# zip prefixes keep state and zip internally consistent, which Faker alone won't do
STATE_ZIP_PREFIX = {
    "VA": "232", "MD": "208", "DC": "200", "NC": "275",
    "PA": "191", "NY": "100", "CA": "941", "TX": "787",
}
STATE_W = [0.44, 0.12, 0.10, 0.08, 0.06, 0.06, 0.07, 0.07]
APT_RATE = 0.25  # share of addresses with a unit/suite line

# ---- Store hours: open daily 10:00-18:00 ----
STORE_OPEN_HOUR, STORE_CLOSE_HOUR = 10, 18
INSTORE_HOURS = list(range(STORE_OPEN_HOUR, STORE_CLOSE_HOUR))
INSTORE_HOUR_W = [0.06, 0.09, 0.12, 0.13, 0.12, 0.15, 0.18, 0.15]  # lunch bump, late-day peak
INSTORE_DOW_W = [0.10, 0.10, 0.12, 0.13, 0.16, 0.23, 0.16]         # Mon..Sun, Saturday busiest

# ---- Online traffic shape: 24/7, evening peak, overnight trough ----
ONLINE_HOURS = list(range(24))
ONLINE_HOUR_W = [
    0.030, 0.018, 0.010, 0.007, 0.006, 0.008, 0.014, 0.022,  # 00-07
    0.030, 0.035, 0.038, 0.042, 0.055, 0.045, 0.040, 0.040,  # 08-15
    0.042, 0.048, 0.060, 0.085, 0.100, 0.095, 0.070, 0.050,  # 16-23
]
ONLINE_DOW_W = [0.15, 0.13, 0.13, 0.13, 0.14, 0.15, 0.17]

# ---- Categorical vocabularies ----
GENDERS = ["female", "male"]
ACQ_SOURCES = ["organic", "mailchimp", "social", "referral"]
DEVICES = ["mobile", "desktop", "tablet"]
DEVICE_W = [0.62, 0.30, 0.08]
LANDING_PAGES = ["/", "/new-arrivals", "/sale", "/collections/best-sellers", "/product"]
EVENT_TYPES = ["page_view", "product_view", "add_to_cart", "checkout_start", "purchase"]

PAYMENT_METHODS = {
    "in_store": (["card_present", "cash", "apple_pay", "gift_card"], [0.60, 0.20, 0.15, 0.05]),
    "online": (["card_not_present", "paypal", "apple_pay"], [0.65, 0.20, 0.15]),
}
DECLINE_RATE = {"in_store": 0.02, "online": 0.05}  # card-present rarely fails
BASKET_SIZE = {
    "in_store": ([1, 2, 3, 4, 5], [0.30, 0.32, 0.20, 0.12, 0.06]),
    "online": ([1, 2, 3], [0.62, 0.28, 0.10]),
}

ENTRY_METHODS = ["chip", "tap", "swipe"]
ENTRY_METHOD_W = [0.50, 0.42, 0.08]
REGISTERS = ["REG01", "REG02"]
EMPLOYEES = ["EMP01", "EMP02", "EMP03", "EMP04"]
PROMO_CODES = ["WELCOME10", "SUMMER5", "MAILCHIMP15"]
FREE_SHIP_THRESHOLD = 75.0
SHIPPING_FEE = 6.95

repeat_customers = round(N_CUSTOMERS * TARGET_REPEAT)
print(f"Config: {N_CUSTOMERS} customers ({repeat_customers} repeat buyers), {N_PRODUCTS} products")
print(f"History {HISTORY_START.date()} -> {TODAY.date()}  |  site launched {LAUNCH_DATE.date()}")


Config: 15 customers (3 repeat buyers), 30 products
History 2025-02-28 -> 2026-08-31  |  site launched 2026-05-31


## Timestamp sampling

The single most visible tell of synthetic transaction data is uniformly-spread timestamps.
`sample_datetime` draws a day weighted by day-of-week, then an hour weighted by time-of-day,
using a different shape per channel. In-store draws are confined to store hours, and the
closing hour is tapered because nobody walks in at 17:55.


In [77]:
def _weighted_days(start, end, dow_w):
    days = pd.date_range(pd.Timestamp(start).normalize(), pd.Timestamp(end).normalize(), freq="D")
    w = np.array([dow_w[d.weekday()] for d in days], dtype=float)
    return days, w / w.sum()


def sample_datetime(channel, start, end):
    """Draw an order timestamp with channel-appropriate day-of-week and time-of-day shape."""
    dow_w = INSTORE_DOW_W if channel == "in_store" else ONLINE_DOW_W
    days, day_p = _weighted_days(start, end, dow_w)
    day = days[np.random.choice(len(days), p=day_p)]

    if channel == "in_store":
        hour_w = np.array(INSTORE_HOUR_W) / sum(INSTORE_HOUR_W)
        hour = int(np.random.choice(INSTORE_HOURS, p=hour_w))
        # taper the closing hour: no walk-ins in the last 15 minutes
        minute = int(np.random.randint(0, 45 if hour == STORE_CLOSE_HOUR - 1 else 60))
    else:
        hour_w = np.array(ONLINE_HOUR_W) / sum(ONLINE_HOUR_W)
        hour = int(np.random.choice(ONLINE_HOURS, p=hour_w))
        minute = int(np.random.randint(0, 60))

    return day + pd.Timedelta(hours=hour, minutes=minute, seconds=int(np.random.randint(0, 60)))


_demo = pd.Series([sample_datetime("in_store", HISTORY_START, TODAY).hour for _ in range(500)])
print("In-store hour range:", _demo.min(), "-", _demo.max(), "(store open 10-18)")


In-store hour range: 10 - 17 (store open 10-18)


## Address sampling

Addresses are generated as separate components (`line1` / `line2` / `city` / `state` / `zip`)
rather than one blob, so location analysis can group by state or zip without parsing.

Two realism details: customers cluster near the store rather than being spread uniformly over
all 50 states and territories, and the **zip prefix is derived from the state** — Faker's
`state_abbr()` and `zipcode()` are independent, so used naively they happily produce a Virginia
address with a Montana zip.


In [78]:
def sample_address(prefix=""):
    """A US address as separate components. `prefix` namespaces the keys, e.g. 'shipping_'."""
    state = str(np.random.choice(list(STATE_ZIP_PREFIX), p=STATE_W))
    zip_code = f"{STATE_ZIP_PREFIX[state]}{np.random.randint(0, 100):02d}"
    return {
        # building_number + street_name, not street_address() — the latter embeds a unit
        # number, which would collide with address_line2
        f"{prefix}address_line1": f"{fake.building_number()} {fake.street_name()}",
        f"{prefix}address_line2": fake.secondary_address() if random.random() < APT_RATE else None,
        f"{prefix}city": fake.city(),
        f"{prefix}state": state,
        f"{prefix}zip": zip_code,
    }


def copy_address(row, prefix="shipping_"):
    """Re-key a customer's address onto an order as the shipping address."""
    return {f"{prefix}{f}": row[f"{f}"] for f in
            ("address_line1", "address_line2", "city", "state", "zip")}


for _ in range(3):
    print(sample_address())


{'address_line1': '104 Jeffrey Street', 'address_line2': None, 'city': 'Lake Joyside', 'state': 'TX', 'zip': '78721'}
{'address_line1': '96001 Jesse Rapids', 'address_line2': 'Apt. 838', 'city': 'Robinsonshire', 'state': 'NY', 'zip': '10085'}
{'address_line1': '0265 Moore Track', 'address_line2': None, 'city': 'Curtisfurt', 'state': 'DC', 'zip': '20025'}


## Table 4 — `products`  [PK: product_id]
Built first because orders/order_items and events reference it.

CapFinch is a boutique carrying 38 SKUs across 5 main categories and 19 subcategories:

| Category | Subcategory | Price Range |
|---|---|---|
| Apparel | Tops, Dresses, Outerwear, Bottoms, Denim | $35–$250 |
| Accessories | Jewelry (everyday), Jewelry (statement), Handbags, Scarves & wraps, Belts, Sunglasses | $20–$220 |
| Footwear | Sandals/flats, Boots, Heels | $50–$220 |
| Home & Gift | Candles, Small home decor, Gift sets | $18–$75 |
| Beauty/Wellness | Skincare, Fragrance | $20–$95 |

Prices and costs end in `.99` or round numbers (`.00`). Stock depth is inverse to price — cheap impulse items are stocked deep, high-ticket anchors are stocked thin.


In [ ]:
# (product_name, category, subcategory, price, cost, stock_on_hand)
CATALOG = [
    ("Ribbed Cotton Knit Tee", "Apparel", "Tops", 38.00, 15.00, 24),
    ("Silk Button-Down Blouse", "Apparel", "Tops", 78.00, 31.00, 12),
    ("Linen Midi Wrap Dress", "Apparel", "Dresses", 128.00, 51.00, 8),
    ("Tiered Floral Maxi Dress", "Apparel", "Dresses", 149.99, 60.00, 6),
    ("Tailored Wool Cardigan", "Apparel", "Outerwear", 120.00, 48.00, 6),
    ("Structured Utility Jacket", "Apparel", "Outerwear", 225.00, 90.00, 4),
    ("Pleated High-Waist Trousers", "Apparel", "Bottoms", 88.00, 35.00, 10),
    ("A-Line Midi Skirt", "Apparel", "Bottoms", 68.00, 27.00, 12),
    ("Straight-Leg Ankle Denim", "Apparel", "Denim", 118.00, 47.00, 14),
    ("Wide-Leg High-Rise Jean", "Apparel", "Denim", 139.99, 56.00, 10),

    ("Gold Vermeil Hoop Earrings", "Accessories", "Jewelry (everyday)", 48.00, 19.00, 18),
    ("Minimalist Chain Necklace", "Accessories", "Jewelry (everyday)", 35.00, 14.00, 22),
    ("Freshwater Pearl Drop Earrings", "Accessories", "Jewelry (statement/special occasion)", 85.00, 34.00, 8),
    ("Chunky Statement Cuff", "Accessories", "Jewelry (statement/special occasion)", 98.00, 39.00, 6),
    ("Leather Crossbody Bag", "Accessories", "Handbags", 145.00, 58.00, 8),
    ("Canvas Carryall Tote", "Accessories", "Handbags", 95.00, 38.00, 12),
    ("Silk Twill Scarf", "Accessories", "Scarves & wraps", 58.00, 23.00, 15),
    ("Cashmere Blend Wrap", "Accessories", "Scarves & wraps", 68.00, 27.00, 10),
    ("Classic Leather Belt", "Accessories", "Belts", 45.00, 18.00, 16),
    ("Woven Waist Belt", "Accessories", "Belts", 32.00, 12.00, 18),
    ("Cat-Eye Acetate Sunglasses", "Accessories", "Sunglasses", 65.00, 26.00, 14),
    ("Classic Aviator Sunglasses", "Accessories", "Sunglasses", 48.00, 19.00, 16),

    ("Leather Slide Sandals", "Footwear", "Sandals/flats", 68.00, 27.00, 12),
    ("Pointed-Toe Ballet Flats", "Footwear", "Sandals/flats", 85.00, 34.00, 10),
    ("Ankle Leather Chelsea Boots", "Footwear", "Boots", 165.00, 66.00, 6),
    ("Suede Tall Riding Boots", "Footwear", "Boots", 210.00, 84.00, 4),
    ("Strappy Block-Heel Pumps", "Footwear", "Heels", 98.00, 39.00, 8),
    ("Classic Kitten Heels", "Footwear", "Heels", 88.00, 35.00, 10),

    ("Soy Wax Signature Candle", "Home & Gift", "Candles", 28.00, 11.00, 24),
    ("Botanical Glass Candle", "Home & Gift", "Candles", 34.00, 13.00, 18),
    ("Speckled Ceramic Vase", "Home & Gift", "Small home decor", 42.00, 18.00, 10),
    ("Brass Picture Frame", "Home & Gift", "Small home decor", 35.00, 15.00, 14),
    ("Self-Care Bath Gift Set", "Home & Gift", "Gift sets", 58.00, 26.00, 12),
    ("Artisanal Tea & Mug Set", "Home & Gift", "Gift sets", 45.00, 20.00, 15),

    ("Hydrating Facial Oil", "Beauty/Wellness", "Skincare", 38.00, 13.00, 16),
    ("Nourishing Botanical Cleanser", "Beauty/Wellness", "Skincare", 27.99, 9.00, 20),
    ("Eau de Parfum Travel Spray", "Beauty/Wellness", "Fragrance", 52.00, 18.00, 14),
    ("Botanical Roll-On Perfume Oil", "Beauty/Wellness", "Fragrance", 68.00, 23.00, 12),
]


def price_band(p):
    if p < 50:
        return "<$50"
    if p < 100:
        return "$50-100"
    if p < 200:
        return "$100-200"
    return "$200+"


products = []
for i, (name, cat, subcat, price, cost, stock) in enumerate(CATALOG[:N_PRODUCTS], start=1):
    products.append({
        "product_id": f"PROD{i:04d}",
        "product_name": name,
        "category": cat,
        "subcategory": subcat,
        "price": price,
        "price_band": price_band(price),
        "cost": cost,
        "stock_on_hand": stock,
    })

products_df = pd.DataFrame(products)

print(f"{len(products_df)} SKUs across {products_df['category'].nunique()} categories")
print(f"Units on hand: {products_df['stock_on_hand'].sum()} total, "
      f"median {int(products_df['stock_on_hand'].median())} per SKU")
print(
    products_df.assign(margin=1 - products_df["cost"] / products_df["price"])
    .groupby(["category", "subcategory"])
    .agg(skus=("product_id", "count"), low=("price", "min"), high=("price", "max"), avg_margin=("margin", "mean"))
    .round(2)
    .to_string()
)

products_df


30 SKUs across 6 categories
Units on hand: 380 total, median 12 per SKU
                 skus    low   high  avg_margin
category                                       
Accessories         5  24.95  85.00        0.63
Bath & Body         5   9.00  39.50        0.56
Home                5  25.95  98.00        0.56
Kitchen & Table     5  13.00  85.95        0.48
Pantry & Treats     5   6.00  25.00        0.39
Stationery          5   6.00  36.00        0.50


,product_id,product_name,category,price,price_band,cost,stock_on_hand
0,PROD0001,Letterpress Greeting Card,Stationery,6.00,<$25,2.95,15
1,PROD0002,A5 Linen Notebook,Stationery,18.95,<$25,8.98,10
2,PROD0003,Brass Fountain Pen,Stationery,36.00,$25-75,19.26,9
3,PROD0004,Weekly Desk Planner,Stationery,18.50,<$25,9.48,6
4,PROD0005,Washi Tape Trio,Stationery,9.00,<$25,4.26,17
5,PROD0006,Soy Wax Candle,Home,28.95,$25-75,11.86,7
6,PROD0007,Speckled Ceramic Vase,Home,44.50,$25-75,20.23,12
7,PROD0008,Linen Throw Blanket,Home,98.00,$75+,39.77,1
8,PROD0009,Brass Picture Frame,Home,25.95,$25-75,12.18,5
9,PROD0010,Reed Diffuser,Home,38.00,$25-75,17.95,13


### Sell-through weights (best sellers)

Real boutique sales are heavily Pareto — a handful of SKUs drive most units. Picking products
uniformly would flatten that and make top-seller and 80/20 analysis meaningless, so every SKU gets
a **popularity weight** built from three things:

1. **Price elasticity** — cheap impulse items outsell expensive anchors, as `(median_price / price) ** 0.6`
2. **Hero boost** — a few deliberate best sellers get a multiplier, so the ranking isn't purely
   "cheapest wins" (the gold hoops sell well *despite* being one of the pricier SKUs)
3. **Taste jitter** — a lognormal wobble so the order isn't perfectly predictable from price

These weights drive both what gets bought (`order_items`) and what gets browsed (`product_view`
events). They are a generator input, not a column on `products` — Square's catalog wouldn't
export them.


In [ ]:
# SKUs that sell above what price alone would predict
HERO_SKUS = {
    "Soy Wax Signature Candle",
    "Gold Vermeil Hoop Earrings",
    "Silk Button-Down Blouse",
    "Hydrating Facial Oil",
    "Leather Crossbody Bag",
}
PRICE_ELASTICITY = 0.6  # higher = cheap items dominate more
HERO_BOOST = 3.5

_w = (products_df["price"].median() / products_df["price"]) ** PRICE_ELASTICITY
_w = _w * np.where(products_df["product_name"].isin(HERO_SKUS), HERO_BOOST, 1.0)
_w = _w * np.exp(np.random.normal(0, 0.35, len(products_df)))
product_weights = (_w / _w.sum()).to_numpy()

print("Top 8 by expected sell-through")
print(
    products_df.assign(share=product_weights)
    .nlargest(8, "share")[["product_name", "category", "price", "share"]]
    .assign(share=lambda d: (d["share"] * 100).round(1).astype(str) + "%")
    .to_string(index=False)
)


Top 8 by expected sell-through
             product_name        category  price share
            Stoneware Mug Kitchen & Table  20.00 10.1%
Letterpress Greeting Card      Stationery   6.00  8.5%
           Soy Wax Candle            Home  28.95  8.5%
          Washi Tape Trio      Stationery   9.00  8.3%
          Shea Hand Cream     Bath & Body  25.00  8.1%
       Gold Vermeil Hoops     Accessories  51.95  5.7%
   Sea Salt Chocolate Bar Pantry & Treats   6.00  4.8%
        A5 Linen Notebook      Stationery  18.95  3.8%


## Table 1 — `customers`  [PK: customer_id]
Only people CapFinch can actually identify (Square Customer Directory / Mailchimp audience).
`total_orders` is assigned here and counts **attributable** orders only — repeat buyers get 2.
Anonymous walk-ins never appear in this table. `signup_date` is drawn inside the history
window so every customer exists before their first order; `first_order_date` is filled in
from the actual orders once they exist.

**`birthdate` is optional and is the source of truth for age.** Handing over a birthday is
opt-in at signup, so only ~45% of customers have one; `age` and `age_band` are derived from it
and are **null for everyone else**. Any age-based analysis has to cope with that gap rather than
assume full coverage.


In [81]:
def age_band(a):
    if a is None:
        return None
    if a <= 24:
        return "18-24"
    if a <= 34:
        return "25-34"
    if a <= 44:
        return "35-44"
    return "45+"


# order-count per customer: repeat buyers get 2 orders, everyone else gets 1
order_counts = [2] * repeat_customers + [1] * (N_CUSTOMERS - repeat_customers)
random.shuffle(order_counts)

signup_window = (TODAY - pd.Timedelta(days=30) - HISTORY_START).days

customers = []
for i in range(1, N_CUSTOMERS + 1):
    # birthday is opt-in at signup, so most customers have no age on file
    if random.random() < BIRTHDATE_CAPTURE_RATE:
        birthdate = fake.date_between(start_date="-66y", end_date="-18y")
        age = int((TODAY.date() - birthdate).days // 365.25)
    else:
        birthdate, age = None, None

    signup = HISTORY_START + pd.Timedelta(days=int(np.random.randint(0, signup_window)))
    customers.append({
        "customer_id": f"CUST{i:04d}",
        "email": fake.unique.email(),
        "birthdate": birthdate,
        "age": age,
        "age_band": age_band(age),
        "gender": random.choice(GENDERS),
        **sample_address(),                # default / billing address
        "signup_date": signup.date(),
        "acquisition_source": random.choice(ACQ_SOURCES),
        "first_order_date": pd.NaT,        # filled after orders are built
        "total_orders": order_counts[i - 1],
    })

customers_df = pd.DataFrame(customers)
customers_df["birthdate"] = pd.to_datetime(customers_df["birthdate"])
customers_df["age"] = customers_df["age"].astype("Int64")

have_bday = customers_df["birthdate"].notna().sum()
print(f"{have_bday}/{len(customers_df)} customers have a birthday on file "
      f"({have_bday / len(customers_df):.0%}) — age/age_band are null for the rest")
print(f"Top states: {customers_df['state'].value_counts().head(4).to_dict()}")
customers_df


6/15 customers have a birthday on file (40%) — age/age_band are null for the rest
Top states: {'VA': 5, 'DC': 5, 'NC': 2, 'MD': 1}


,customer_id,email,birthdate,age,age_band,gender,address_line1,address_line2,city,state,zip,signup_date,acquisition_source,first_order_date,total_orders
0,CUST0001,howardmaurice@example.com,NaT,<NA>,NaN,male,78161 Calderon River,Suite 931,Lake Jeremyport,MD,20831,2025-08-19,social,NaT,1
1,CUST0002,wyattmichelle@example.com,NaT,<NA>,NaN,female,7525 Clark Grove,NaN,New Cynthiaside,VA,23212,2025-07-05,mailchimp,NaT,1
2,CUST0003,frankgray@example.net,NaT,<NA>,NaN,male,835 Jeremy Bypass,NaN,Richardland,DC,20092,2026-05-20,mailchimp,NaT,1
3,CUST0004,robinbradley@example.net,NaT,<NA>,NaN,male,3767 Joseph Gateway,Suite 969,Coxberg,DC,20059,2025-04-11,referral,NaT,1
4,CUST0005,camposmichelle@example.org,1985-02-13,41,35-44,male,269 Paul Ranch,NaN,Riceside,VA,23204,2026-06-09,organic,NaT,1
5,CUST0006,frazierdanny@example.net,NaT,<NA>,NaN,female,5146 Shawn Stravenue,NaN,Teresaburgh,VA,23288,2026-06-09,referral,NaT,1
6,CUST0007,harrellkenneth@example.net,1985-01-12,41,35-44,male,932 Farmer Plains,NaN,New Mariotown,TX,78730,2026-03-16,social,NaT,2
7,CUST0008,hickmannatasha@example.com,NaT,<NA>,NaN,female,03911 Cabrera Trace,Apt. 278,West Allison,VA,23286,2025-04-24,mailchimp,NaT,1
8,CUST0009,ibrandt@example.net,NaT,<NA>,NaN,female,346 Kim Path,NaN,Martinezbury,VA,23220,2025-05-21,organic,NaT,1
9,CUST0010,chapmanjerry@example.org,1963-09-27,62,45+,male,10310 Jones Freeway,NaN,Elizabethborough,PA,19106,2026-01-22,social,NaT,1


## Tables 2 & 3 — `orders` and `order_items`
One `orders` row per completed sale from **either** channel, distinguished by `channel`.
Channel-specific columns stay null on the other side:

- **online only** — `session_id`, `shipping_state`, `shipping_zip`, `shipping_fee`,
  `fulfillment_type`, `promo_code`, `device`
- **in-store only** — `register_id`, `employee_id`, `entry_method`, `tip_amount`, `receipt_type`

Money: `order_total = subtotal − discount_amount + shipping_fee`, where `subtotal` is the sum
of the order's `order_items.line_total`.

Orders are built in two passes: first the attributable ones expanded from each customer's
`total_orders`, then anonymous walk-ins and guest checkouts layered on top to hit the
identity-capture rates. Only online orders create a converting session.


In [87]:
order_rows, oi_rows, sess_rows = [], [], []
order_counter, session_counter = count(1), count(1)

product_ids = products_df["product_id"].tolist()
price_lookup = products_df.set_index("product_id")["price"].to_dict()

SHIP_FIELDS = ["shipping_address_line1", "shipping_address_line2",
               "shipping_city", "shipping_state", "shipping_zip"]


def build_order(customer, channel, when):
    """One order plus its line items. `customer` is None for a walk-in or guest checkout."""
    tid = f"TXN{next(order_counter):05d}"

    sizes, size_p = BASKET_SIZE[channel]
    n_items = min(int(np.random.choice(sizes, p=size_p)), len(product_ids))
    chosen = np.random.choice(product_ids, size=n_items, replace=False, p=product_weights)

    subtotal, total_qty = 0.0, 0
    for pid in chosen:
        qty = int(np.random.randint(1, 3 if channel == "online" else 4))
        unit = float(price_lookup[pid])
        line = round(qty * unit, 2)
        subtotal += line
        total_qty += qty
        oi_rows.append({
            "order_item_id": f"OI{len(oi_rows) + 1:06d}",
            "transaction_id": tid,
            "product_id": str(pid),
            "quantity": qty,
            "unit_price": unit,
            "line_total": line,
        })
    subtotal = round(subtotal, 2)

    methods, method_p = PAYMENT_METHODS[channel]
    method = str(np.random.choice(methods, p=method_p))
    declined = method != "cash" and random.random() < DECLINE_RATE[channel]  # cash never declines
    discount = round(subtotal * np.random.choice([0, 0.05, 0.10], p=[0.7, 0.2, 0.1]), 2)

    row = {
        "transaction_id": tid,
        "customer_id": customer["customer_id"] if customer is not None else None,
        "channel": channel,
        "order_datetime": when,
        "order_date": when.date(),
        "day_of_week": when.day_name(),
        "hour_of_day": when.hour,
        "subtotal": subtotal,
        "discount_amount": discount,
        "item_count": total_qty,
        "payment_method": method,
        "payment_status": "declined" if declined else "authorized",
        # online-only
        "session_id": None,
        **{f: None for f in SHIP_FIELDS},
        "is_gift_ship": None,
        "shipping_fee": None,
        "fulfillment_type": None,
        "promo_code": None,
        "device": None,
        # in-store-only
        "register_id": None,
        "employee_id": None,
        "entry_method": None,
        "tip_amount": None,
        "receipt_type": None,
    }

    if channel == "online":
        sid = f"SESS{next(session_counter):06d}"
        fulfillment = str(np.random.choice(["ship", "pickup_in_store"], p=[0.85, 0.15]))
        free_ship = fulfillment == "pickup_in_store" or subtotal >= FREE_SHIP_THRESHOLD
        device = str(np.random.choice(DEVICES, p=DEVICE_W))

        # gift orders ship somewhere other than the buyer's own address
        gift = customer is None or random.random() < GIFT_SHIP_RATE
        ship_to = sample_address("shipping_") if gift else copy_address(customer)

        row.update({
            "session_id": sid,
            **ship_to,
            "is_gift_ship": gift,
            "shipping_fee": 0.0 if free_ship else SHIPPING_FEE,
            "fulfillment_type": fulfillment,
            "promo_code": random.choice(PROMO_CODES) if discount > 0 else None,
            "device": device,
        })
        sess_rows.append({
            "session_id": sid,
            "customer_id": row["customer_id"],
            "session_start": when - pd.Timedelta(minutes=int(np.random.randint(3, 30))),
            "device": device,
            "traffic_source": customer["acquisition_source"] if customer is not None else random.choice(ACQ_SOURCES),
            "landing_page": random.choice(LANDING_PAGES),
            "reached_cart": True,
            "converted": True,
        })
    else:
        row.update({
            "register_id": random.choice(REGISTERS),
            "employee_id": random.choice(EMPLOYEES),
            "entry_method": None if method == "cash" else str(np.random.choice(ENTRY_METHODS, p=ENTRY_METHOD_W)),
            "tip_amount": 0.0 if random.random() < 0.9 else round(subtotal * 0.05, 2),
            # a digital receipt is what links a walk-in to a customer record
            "receipt_type": random.choice(["email", "sms"]) if customer is not None else random.choice(["printed", "none"]),
        })

    row["order_total"] = round(subtotal - discount + (row["shipping_fee"] or 0.0), 2)
    return row


# Pass 1 — attributable orders. The channel split is allocated exactly rather than drawn
# per order, so the mix still holds at small N.
attributable = [cust for _, cust in customers_df.iterrows() for _ in range(int(cust["total_orders"]))]
n_attr_online = round(len(attributable) * P_ONLINE_ATTRIBUTABLE)
attr_channels = ["online"] * n_attr_online + ["in_store"] * (len(attributable) - n_attr_online)
random.shuffle(attr_channels)

for cust, channel in zip(attributable, attr_channels):
    signup = pd.Timestamp(cust["signup_date"])
    window_start = max(signup, LAUNCH_DATE) if channel == "online" else max(signup, HISTORY_START)
    order_rows.append(build_order(cust, channel, sample_datetime(channel, window_start, TODAY)))

# Pass 2 — anonymous walk-ins and guest checkouts, sized to hit the identity-capture rates
n_attr_instore = len(attributable) - n_attr_online
n_anon_instore = round(n_attr_instore * INSTORE_ANON_RATE / (1 - INSTORE_ANON_RATE))
n_guest_online = round(n_attr_online * ONLINE_GUEST_RATE / (1 - ONLINE_GUEST_RATE))

for _ in range(n_anon_instore):
    order_rows.append(build_order(None, "in_store", sample_datetime("in_store", HISTORY_START, TODAY)))
for _ in range(n_guest_online):
    order_rows.append(build_order(None, "online", sample_datetime("online", LAUNCH_DATE, TODAY)))

orders_df = pd.DataFrame(order_rows).sort_values("order_datetime").reset_index(drop=True)
# object dtype would make `~is_gift_ship` invert to ints rather than negate
orders_df["is_gift_ship"] = orders_df["is_gift_ship"].astype("boolean")
order_items_df = pd.DataFrame(oi_rows)

online_mask = orders_df["channel"] == "online"
print(f"orders={len(orders_df)}  order_items={len(order_items_df)}")
print(orders_df["channel"].value_counts().to_string())
print(f"anonymous in-store={n_anon_instore}  guest online={n_guest_online}")
print(f"online orders run {orders_df.loc[online_mask, 'order_date'].min()} "
      f"-> {orders_df.loc[online_mask, 'order_date'].max()}")
print(f"shipped to an address other than the buyer's: {int(orders_df['is_gift_ship'].sum())} of {online_mask.sum()}")

# rows are date-sorted, so every online order sits at the tail — preview both channels
pd.concat([
    orders_df[~online_mask].head(6),
    orders_df[online_mask].head(6),
])


orders=42  order_items=93
channel
in_store    34
online       8
anonymous in-store=23  guest online=1
online orders run 2026-06-04 -> 2026-08-17
shipped to an address other than the buyer's: 3 of 8


,transaction_id,customer_id,channel,order_datetime,order_date,day_of_week,hour_of_day,subtotal,discount_amount,item_count,payment_method,payment_status,session_id,shipping_address_line1,shipping_address_line2,shipping_city,shipping_state,shipping_zip,is_gift_ship,shipping_fee,fulfillment_type,promo_code,device,register_id,employee_id,entry_method,tip_amount,receipt_type,order_total
0,TXN00024,NaN,in_store,2025-04-11 14:32:27,2025-04-11,Friday,14,12.00,0.00,2,cash,authorized,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,REG01,EMP02,NaN,0.0,printed,12.00
1,TXN00019,NaN,in_store,2025-04-25 12:16:15,2025-04-25,Friday,12,70.45,0.00,2,card_present,authorized,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,REG01,EMP02,chip,0.0,none,70.45
2,TXN00010,CUST0009,in_store,2025-06-16 15:39:28,2025-06-16,Monday,15,12.00,1.20,2,card_present,authorized,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,REG02,EMP03,tap,0.0,email,10.80
3,TXN00029,NaN,in_store,2025-07-14 17:17:20,2025-07-14,Monday,17,202.85,0.00,7,cash,authorized,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,REG01,EMP02,NaN,0.0,none,202.85
4,TXN00039,NaN,in_store,2025-07-25 17:30:50,2025-07-25,Friday,17,69.50,0.00,2,card_present,authorized,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,REG02,EMP02,tap,0.0,none,69.50
5,TXN00033,NaN,in_store,2025-08-07 17:37:41,2025-08-07,Thursday,17,118.90,5.94,6,card_present,authorized,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,REG01,EMP01,chip,0.0,none,112.96
25,TXN00042,NaN,online,2026-06-04 19:57:04,2026-06-04,Thursday,19,72.95,3.65,5,card_not_present,authorized,SESS000008,16400 Wise Route,Suite 278,Hoffmanville,DC,20050,True,6.95,ship,SUMMER5,mobile,NaN,NaN,NaN,NaN,NaN,76.25
28,TXN00004,CUST0004,online,2026-06-20 15:37:45,2026-06-20,Saturday,15,18.95,0.00,1,card_not_present,authorized,SESS000002,808 Huynh Cove,NaN,West Darrell,TX,78724,True,6.95,ship,NaN,desktop,NaN,NaN,NaN,NaN,NaN,25.90
29,TXN00016,CUST0014,online,2026-06-23 22:06:48,2026-06-23,Tuesday,22,45.45,4.55,2,card_not_present,authorized,SESS000007,98050 Breanna Parkway,NaN,North Susan,DC,20037,False,6.95,ship,SUMMER5,tablet,NaN,NaN,NaN,NaN,NaN,47.85
31,TXN00012,CUST0011,online,2026-07-02 08:45:01,2026-07-02,Thursday,8,56.90,0.00,4,card_not_present,authorized,SESS000005,82449 Jennifer Ford,NaN,Williamview,DC,20065,True,6.95,ship,NaN,mobile,NaN,NaN,NaN,NaN,NaN,63.85


In [14]:
# order_items preview
order_items_df.head(15)

,order_item_id,transaction_id,product_id,quantity,unit_price,line_total
0,OI000001,TXN00001,PROD0004,2,58.18,116.36
1,OI000002,TXN00001,PROD0003,2,39.14,78.28
2,OI000003,TXN00001,PROD0005,2,57.24,114.48
3,OI000004,TXN00002,PROD0005,3,57.24,171.72
4,OI000005,TXN00003,PROD0006,3,50.66,151.98
5,OI000006,TXN00004,PROD0008,2,110.13,220.26
6,OI000007,TXN00004,PROD0006,1,50.66,50.66
7,OI000008,TXN00004,PROD0009,3,87.09,261.27
8,OI000009,TXN00005,PROD0006,2,50.66,101.32
9,OI000010,TXN00006,PROD0005,2,57.24,114.48


## Table 5 — `sessions`  [PK: session_id]
Website visits only — a POS has no equivalent, which is exactly why the funnel KPIs need this
table. The converting sessions already exist (one per **online** order). Here we add
**abandoned-cart** and **browse-only** sessions so the online funnel hits conversion ~2% and
abandonment ~68%:

- `total_carts = online_orders / (1 − 0.68)` → abandoned = carts − online orders
- `total_sessions = online_orders / 0.02` → browse-only = sessions − carts

All sessions fall on or after `LAUNCH_DATE`, and some are anonymous (`customer_id = None`).
In-store orders are deliberately excluded from both sides of the conversion ratio.


In [88]:
n_online_orders = int((orders_df["channel"] == "online").sum())
total_carts = round(n_online_orders / (1 - TARGET_ABANDONMENT))
abandoned_carts = max(total_carts - n_online_orders, 0)
total_sessions = round(n_online_orders / TARGET_CONVERSION)
browse_sessions = max(total_sessions - total_carts, 0)

cust_ids = customers_df["customer_id"].tolist()


def make_session(reached_cart, converted):
    # ~40% of non-converting traffic is a known customer, the rest is anonymous
    cust_id = random.choice(cust_ids) if random.random() < 0.4 else None
    return {
        "session_id": f"SESS{next(session_counter):06d}",
        "customer_id": cust_id,
        "session_start": sample_datetime("online", LAUNCH_DATE, TODAY),
        "device": str(np.random.choice(DEVICES, p=DEVICE_W)),
        "traffic_source": random.choice(ACQ_SOURCES),
        "landing_page": random.choice(LANDING_PAGES),
        "reached_cart": reached_cart,
        "converted": converted,
    }


for _ in range(abandoned_carts):
    sess_rows.append(make_session(reached_cart=True, converted=False))
for _ in range(browse_sessions):
    sess_rows.append(make_session(reached_cart=False, converted=False))

sessions_df = pd.DataFrame(sess_rows).sort_values("session_start").reset_index(drop=True)
print(f"sessions={len(sessions_df)}  carts={total_carts}  abandoned={abandoned_carts}  browse={browse_sessions}")
sessions_df.head(15)


sessions=400  carts=25  abandoned=17  browse=375


,session_id,customer_id,session_start,device,traffic_source,landing_page,reached_cart,converted
0,SESS000188,NaN,2026-05-31 00:09:23,mobile,organic,/sale,False,False
1,SESS000256,CUST0010,2026-05-31 11:19:36,mobile,mailchimp,/product,False,False
2,SESS000181,NaN,2026-05-31 11:21:33,desktop,social,/,False,False
3,SESS000274,CUST0014,2026-05-31 11:51:14,mobile,organic,/collections/best-sellers,False,False
4,SESS000266,CUST0004,2026-06-01 10:24:58,desktop,social,/new-arrivals,False,False
5,SESS000030,NaN,2026-06-01 15:05:38,mobile,mailchimp,/product,False,False
6,SESS000187,NaN,2026-06-01 18:01:29,mobile,social,/new-arrivals,False,False
7,SESS000371,CUST0009,2026-06-01 18:32:12,desktop,mailchimp,/collections/best-sellers,False,False
8,SESS000048,NaN,2026-06-01 20:04:03,mobile,social,/collections/best-sellers,False,False
9,SESS000208,CUST0006,2026-06-01 21:08:01,mobile,mailchimp,/collections/best-sellers,False,False


### Back-fill `first_order_date`
Now that orders exist, set each customer's first order date from their earliest **attributable**
order, across both channels. This is the single source of truth for first-purchase / cohort
analysis — there is no `is_first_order` flag on `orders`, since it is derivable from here and
would be misleading anyway while most in-store sales are anonymous.


In [89]:
first_orders = orders_df.dropna(subset=["customer_id"]).groupby("customer_id")["order_datetime"].min()
customers_df["first_order_date"] = pd.to_datetime(customers_df["customer_id"].map(first_orders)).dt.date
customers_df


,customer_id,email,birthdate,age,age_band,gender,address_line1,address_line2,city,state,zip,signup_date,acquisition_source,first_order_date,total_orders
0,CUST0001,howardmaurice@example.com,NaT,<NA>,NaN,male,78161 Calderon River,Suite 931,Lake Jeremyport,MD,20831,2025-08-19,social,2026-02-20,1
1,CUST0002,wyattmichelle@example.com,NaT,<NA>,NaN,female,7525 Clark Grove,NaN,New Cynthiaside,VA,23212,2025-07-05,mailchimp,2025-11-15,1
2,CUST0003,frankgray@example.net,NaT,<NA>,NaN,male,835 Jeremy Bypass,NaN,Richardland,DC,20092,2026-05-20,mailchimp,2026-07-12,1
3,CUST0004,robinbradley@example.net,NaT,<NA>,NaN,male,3767 Joseph Gateway,Suite 969,Coxberg,DC,20059,2025-04-11,referral,2026-06-20,1
4,CUST0005,camposmichelle@example.org,1985-02-13,41,35-44,male,269 Paul Ranch,NaN,Riceside,VA,23204,2026-06-09,organic,2026-07-16,1
5,CUST0006,frazierdanny@example.net,NaT,<NA>,NaN,female,5146 Shawn Stravenue,NaN,Teresaburgh,VA,23288,2026-06-09,referral,2026-07-07,1
6,CUST0007,harrellkenneth@example.net,1985-01-12,41,35-44,male,932 Farmer Plains,NaN,New Mariotown,TX,78730,2026-03-16,social,2026-03-22,2
7,CUST0008,hickmannatasha@example.com,NaT,<NA>,NaN,female,03911 Cabrera Trace,Apt. 278,West Allison,VA,23286,2025-04-24,mailchimp,2026-08-17,1
8,CUST0009,ibrandt@example.net,NaT,<NA>,NaN,female,346 Kim Path,NaN,Martinezbury,VA,23220,2025-05-21,organic,2025-06-16,1
9,CUST0010,chapmanjerry@example.org,1963-09-27,62,45+,male,10310 Jones Freeway,NaN,Elizabethborough,PA,19106,2026-01-22,social,2026-03-26,1


## Table 6 — `events`  [PK: event_id]
Funnel detail per session. Every session gets a `page_view` + some `product_view`s; sessions
that reached the cart add `add_to_cart` → `checkout_start`, and converters end with `purchase`.


In [90]:
event_rows = []


def add_event(session, etype, t, pid=None):
    event_rows.append({
        "event_id": f"EVT{len(event_rows) + 1:07d}",
        "session_id": session["session_id"],
        "event_type": etype,
        "product_id": pid,
        "event_time": t,
    })


def viewed_product():
    return str(np.random.choice(product_ids, p=product_weights))


for _, s in sessions_df.iterrows():
    t = pd.to_datetime(s["session_start"])
    add_event(s, "page_view", t)

    for _ in range(int(np.random.randint(1, 4))):
        t += pd.Timedelta(seconds=int(np.random.randint(20, 180)))
        add_event(s, "product_view", t, viewed_product())

    if s["reached_cart"]:
        t += pd.Timedelta(seconds=int(np.random.randint(20, 120)))
        add_event(s, "add_to_cart", t, viewed_product())
        t += pd.Timedelta(seconds=int(np.random.randint(20, 120)))
        add_event(s, "checkout_start", t)
        if s["converted"]:
            t += pd.Timedelta(seconds=int(np.random.randint(20, 120)))
            add_event(s, "purchase", t)

events_df = pd.DataFrame(event_rows)
print(f"events={len(events_df)}")
events_df.head(15)


events=1237


,event_id,session_id,event_type,product_id,event_time
0,EVT0000001,SESS000188,page_view,NaN,2026-05-31 00:09:23
1,EVT0000002,SESS000188,product_view,PROD0030,2026-05-31 00:10:47
2,EVT0000003,SESS000188,product_view,PROD0001,2026-05-31 00:13:22
3,EVT0000004,SESS000256,page_view,NaN,2026-05-31 11:19:36
4,EVT0000005,SESS000256,product_view,PROD0006,2026-05-31 11:21:04
5,EVT0000006,SESS000181,page_view,NaN,2026-05-31 11:21:33
6,EVT0000007,SESS000181,product_view,PROD0023,2026-05-31 11:23:14
7,EVT0000008,SESS000181,product_view,PROD0027,2026-05-31 11:25:08
8,EVT0000009,SESS000181,product_view,PROD0025,2026-05-31 11:27:53
9,EVT0000010,SESS000274,page_view,NaN,2026-05-31 11:51:14


## Overview, KPI check & integrity
Confirm the six DataFrames, verify the KPI targets, and assert the numeric, channel, and
timing rules hold — including that no in-store order lands outside store hours and no online
order predates launch.


In [91]:
tables = {
    "customers": customers_df,
    "orders": orders_df,
    "order_items": order_items_df,
    "products": products_df,
    "sessions": sessions_df,
    "events": events_df,
}

print("Table shapes")
for name, df in tables.items():
    print(f"  {name:12s} rows={len(df):5d}  cols={len(df.columns)}")

instore = orders_df[orders_df["channel"] == "in_store"]
online = orders_df[orders_df["channel"] == "online"]

# --- KPI check (funnel metrics are online-only) ---
conversion = len(online) / len(sessions_df)
carts = int(sessions_df["reached_cart"].sum())
abandonment = 1 - sessions_df["converted"].sum() / carts
repeat = (customers_df["total_orders"] >= 2).mean()

print("\nKPIs (actual vs target)")
print(f"  Conversion rate : {conversion:6.2%}  (target ~2%, online orders / sessions)")
print(f"  Cart abandonment: {abandonment:6.2%}  (target ~68%)")
print(f"  Repeat purchase : {repeat:6.2%}  (target ~20%)")

print("\nChannel profile")
print(f"  In-store : {len(instore):4d} orders  AOV ${instore['order_total'].mean():7.2f}  "
      f"anonymous {instore['customer_id'].isna().mean():.0%}")
print(f"  Online   : {len(online):4d} orders  AOV ${online['order_total'].mean():7.2f}  "
      f"guest {online['customer_id'].isna().mean():.0%}")

print("\nShip-to states (online)")
print(online["shipping_state"].value_counts().to_string())

# --- Best sellers / Pareto ---
sales = (
    order_items_df.merge(products_df, on="product_id")
    .groupby(["product_name", "category"])
    .agg(units=("quantity", "sum"), revenue=("line_total", "sum"))
    .sort_values("units", ascending=False)
)
top_share = sales["revenue"].head(round(len(products_df) * 0.2)).sum() / sales["revenue"].sum()

print("\nTop 8 sellers by units")
print(sales.head(8).round(2).to_string())
print(f"\n  Top 20% of SKUs = {top_share:.0%} of revenue (Pareto check)")
print(f"  SKUs with zero sales: {len(products_df) - len(sales)}")
print("\nCategory mix by revenue")
print((sales.groupby("category")["revenue"].sum() / sales["revenue"].sum()).sort_values(ascending=False).round(3).to_string())

# --- Integrity checks ---
assert (order_items_df["line_total"] == (order_items_df["quantity"] * order_items_df["unit_price"]).round(2)).all()
recon = order_items_df.groupby("transaction_id")["line_total"].sum().round(2)
by_tid = orders_df.set_index("transaction_id").loc[recon.index]
assert np.allclose(recon.values, by_tid["subtotal"].values)
assert np.allclose(
    by_tid["order_total"].values,
    (by_tid["subtotal"] - by_tid["discount_amount"] + by_tid["shipping_fee"].fillna(0.0)).round(2).values,
)
assert orders_df["customer_id"].dropna().isin(customers_df["customer_id"]).all()
assert order_items_df["product_id"].isin(products_df["product_id"]).all()

# channel exclusivity
online_only = ["session_id", "shipping_address_line1", "shipping_city", "shipping_state",
               "shipping_zip", "shipping_fee", "fulfillment_type", "device"]
instore_only = ["register_id", "employee_id", "tip_amount", "receipt_type"]
assert instore[online_only].isna().all().all()
assert online[instore_only].isna().all().all()
assert online["session_id"].notna().all()
assert online["session_id"].isin(sessions_df["session_id"]).all()
assert sessions_df.loc[sessions_df["session_id"].isin(online["session_id"]), "converted"].all()
assert int(sessions_df["converted"].sum()) == len(online)

# addresses: state/zip agree, and non-gift orders ship to the buyer's address on file
for df_, state_col, zip_col in [(customers_df, "state", "zip"), (online, "shipping_state", "shipping_zip")]:
    assert df_.apply(lambda r: r[zip_col].startswith(STATE_ZIP_PREFIX[r[state_col]]), axis=1).all()
known_direct = online[online["customer_id"].notna() & ~online["is_gift_ship"]]
merged = known_direct.merge(customers_df, on="customer_id", suffixes=("", "_cust"))
assert (merged["shipping_address_line1"] == merged["address_line1"]).all()
assert (merged["shipping_zip"] == merged["zip"]).all()

# timing rules
assert instore["hour_of_day"].between(STORE_OPEN_HOUR, STORE_CLOSE_HOUR - 1).all()
assert (online["order_datetime"] >= LAUNCH_DATE).all()
assert (orders_df["order_datetime"] <= TODAY + pd.Timedelta(days=1)).all()
assert (customers_df["total_orders"] == orders_df["customer_id"].value_counts().reindex(customers_df["customer_id"]).values).all()

print("\nIntegrity checks passed: money, FKs, channel exclusivity, addresses, store hours, and launch date all consistent.")


Table shapes
  customers    rows=   15  cols=15
  orders       rows=   42  cols=29
  order_items  rows=   93  cols=6
  products     rows=   30  cols=7
  sessions     rows=  400  cols=8
  events       rows= 1237  cols=5

KPIs (actual vs target)
  Conversion rate :  2.00%  (target ~2%, online orders / sessions)
  Cart abandonment: 68.00%  (target ~68%)
  Repeat purchase : 20.00%  (target ~20%)

Channel profile
  In-store :   34 orders  AOV $ 104.71  anonymous 68%
  Online   :    8 orders  AOV $  66.30  guest 12%

Ship-to states (online)
shipping_state
DC    5
VA    2
TX    1

Top 8 sellers by units
                                           units  revenue
product_name              category                       
Letterpress Greeting Card Stationery          25   150.00
Sea Salt Chocolate Bar    Pantry & Treats     16    96.00
Gold Vermeil Hoops        Accessories         14   727.30
Weekly Desk Planner       Stationery          14   259.00
Shea Hand Cream           Bath & Body         12

## Export to CSV (optional)
Everything lives as DataFrames above. Uncomment to write them to a `data/` folder when ready.


In [14]:
# import os
# os.makedirs("data", exist_ok=True)
# for name, df in tables.items():
#     df.to_csv(f"data/{name}.csv", index=False)
# print("Wrote:", ", ".join(f"data/{n}.csv" for n in tables))